# Finance Credit Risk Analysis — Data Generation
**Project 3 | AllenRattler Analytics Portfolio**

This notebook generates four synthetic CSV files that simulate a personal lending portfolio:
- `loans_raw.csv` — 3,600 loan records with 10 intentional data quality issues
- `payment_events_raw.csv` — monthly payment history with 3 data quality issues
- `loan_grades_lookup.csv` — clean loan grade/subgrade reference table
- `geography_lookup.csv` — clean US state to Census region reference table

**Run cells in order (1 through 6). Do not use Run All.**

---
### Known Data Quality Issues — loans_raw.csv
| Code | Issue | Count | Rate |
|---|---|---|---|
| DQ-1 | Duplicate loan records | 100 | 2.7% |
| DQ-2 | Null loan amounts | 300 | 8.1% |
| DQ-3 | Negative interest rates | 80 | 2.2% |
| DQ-4 | Invalid/null credit scores | 150 | 4.1% |
| DQ-5 | DTI values over 100% | 100 | 2.7% |
| DQ-6 | Date sequence errors | 50 | 1.4% |
| DQ-7 | Mixed-case loan_status values | 120 | 3.2% |
| DQ-8 | Invalid state codes | 40 | 1.1% |
| DQ-9 | Null employment_length values | 200 | 5.4% |
| DQ-10 | Grade/subgrade mismatches | 60 | 1.6% |

In [ ]:
# Cell 1: Install required libraries
# Faker is not pre-installed in Colab — manual install required
!pip install faker

In [ ]:
# Cell 2: Imports and global configuration
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import os

fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# ── Output directory ──────────────────────────────────────────────────────────
output_dir = "data/raw"
os.makedirs(output_dir, exist_ok=True)

print("✅ Libraries loaded. Output directory ready.")

In [ ]:
# Cell 3: Generate clean reference/lookup tables
# These become dim_loan_product and dim_geography in the star schema.

# ── Loan Grades Lookup (dim_loan_product) ─────────────────────────────────────
# Loan grades A–G map to risk tiers.
# A = lowest risk / lowest rate; G = highest risk / highest rate.
# Each grade has 5 sub-grades and a corresponding interest rate band.

grade_data = []
grade_rate_bands = {
    "A": (5.0,  8.0),
    "B": (8.5,  12.0),
    "C": (12.5, 16.5),
    "D": (17.0, 22.0),
    "E": (22.5, 27.0),
    "F": (27.5, 31.5),
    "G": (32.0, 36.0)
}
grade_descriptions = {
    "A": "Prime — Very Low Risk",
    "B": "Near-Prime — Low Risk",
    "C": "Standard — Moderate Risk",
    "D": "Sub-Prime — Elevated Risk",
    "E": "High Risk",
    "F": "Very High Risk",
    "G": "Speculative — Highest Risk"
}

for grade, (rate_min, rate_max) in grade_rate_bands.items():
    for sub in range(1, 6):
        subgrade = f"{grade}{sub}"
        step = (rate_max - rate_min) / 4
        rate = round(rate_min + (sub - 1) * step, 2)
        grade_data.append({
            "loan_grade":        grade,
            "loan_subgrade":     subgrade,
            "grade_description": grade_descriptions[grade],
            "base_interest_rate": rate
        })

loan_grades_df = pd.DataFrame(grade_data)
loan_grades_df.to_csv(f"{output_dir}/loan_grades_lookup.csv", index=False)
print(f"✅ loan_grades_lookup.csv — {len(loan_grades_df)} rows")


# ── Geography Lookup (dim_geography) ─────────────────────────────────────────
# All 50 US states mapped to Census Bureau regions and divisions.

geography_data = [
    ("AL","Alabama","South","East South Central"),
    ("AK","Alaska","West","Pacific"),
    ("AZ","Arizona","West","Mountain"),
    ("AR","Arkansas","South","West South Central"),
    ("CA","California","West","Pacific"),
    ("CO","Colorado","West","Mountain"),
    ("CT","Connecticut","Northeast","New England"),
    ("DE","Delaware","South","South Atlantic"),
    ("FL","Florida","South","South Atlantic"),
    ("GA","Georgia","South","South Atlantic"),
    ("HI","Hawaii","West","Pacific"),
    ("ID","Idaho","West","Mountain"),
    ("IL","Illinois","Midwest","East North Central"),
    ("IN","Indiana","Midwest","East North Central"),
    ("IA","Iowa","Midwest","West North Central"),
    ("KS","Kansas","Midwest","West North Central"),
    ("KY","Kentucky","South","East South Central"),
    ("LA","Louisiana","South","West South Central"),
    ("ME","Maine","Northeast","New England"),
    ("MD","Maryland","South","South Atlantic"),
    ("MA","Massachusetts","Northeast","New England"),
    ("MI","Michigan","Midwest","East North Central"),
    ("MN","Minnesota","Midwest","West North Central"),
    ("MS","Mississippi","South","East South Central"),
    ("MO","Missouri","Midwest","West North Central"),
    ("MT","Montana","West","Mountain"),
    ("NE","Nebraska","Midwest","West North Central"),
    ("NV","Nevada","West","Mountain"),
    ("NH","New Hampshire","Northeast","New England"),
    ("NJ","New Jersey","Northeast","Middle Atlantic"),
    ("NM","New Mexico","West","Mountain"),
    ("NY","New York","Northeast","Middle Atlantic"),
    ("NC","North Carolina","South","South Atlantic"),
    ("ND","North Dakota","Midwest","West North Central"),
    ("OH","Ohio","Midwest","East North Central"),
    ("OK","Oklahoma","South","West South Central"),
    ("OR","Oregon","West","Pacific"),
    ("PA","Pennsylvania","Northeast","Middle Atlantic"),
    ("RI","Rhode Island","Northeast","New England"),
    ("SC","South Carolina","South","South Atlantic"),
    ("SD","South Dakota","Midwest","West North Central"),
    ("TN","Tennessee","South","East South Central"),
    ("TX","Texas","South","West South Central"),
    ("UT","Utah","West","Mountain"),
    ("VT","Vermont","Northeast","New England"),
    ("VA","Virginia","South","South Atlantic"),
    ("WA","Washington","West","Pacific"),
    ("WV","West Virginia","South","South Atlantic"),
    ("WI","Wisconsin","Midwest","East North Central"),
    ("WY","Wyoming","West","Mountain"),
]

geography_df = pd.DataFrame(geography_data,
    columns=["state_code","state_name","census_region","census_division"])
geography_df.to_csv(f"{output_dir}/geography_lookup.csv", index=False)
print(f"✅ geography_lookup.csv — {len(geography_df)} rows")

In [ ]:
# Cell 4: Generate loans_raw.csv — primary messy fact dataset
# Target: 3,600 rows generated → ~3,500 after duplicate removal in Power Query
#
# Intentional data quality issues baked in:
#   [DQ-1]  100 fully duplicate rows
#   [DQ-2]  ~300 null loan amounts
#   [DQ-3]  ~80 negative interest rates
#   [DQ-4]  ~150 credit scores out of valid range (<300 or >850) or null
#   [DQ-5]  ~100 DTI values > 100% (mathematically impossible)
#   [DQ-6]  ~50 date sequence errors (issue_date >= first_payment_date)
#   [DQ-7]  Mixed casing in loan_status field
#   [DQ-8]  ~40 invalid/fictional state codes
#   [DQ-9]  ~200 null employment_length values
#   [DQ-10] ~60 grade/subgrade mismatches

# ── Configuration ─────────────────────────────────────────────────────────────
N_CLEAN = 3_500
N_DUPES = 100
VALID_STATES = geography_df["state_code"].tolist()
GRADES = list("ABCDEFG")
PURPOSES = [
    "debt_consolidation", "credit_card", "home_improvement",
    "medical", "small_business", "major_purchase", "car",
    "vacation", "moving", "wedding", "renewable_energy", "educational"
]
LOAN_STATUSES_CLEAN = [
    "Fully Paid", "Charged Off", "Current",
    "Late (31-120 days)", "Default"
]
STATUS_WEIGHTS  = [0.42, 0.18, 0.28, 0.07, 0.05]
TERMS           = [36, 60]
HOME_OWNERSHIP  = ["RENT", "OWN", "MORTGAGE", "OTHER"]
HOME_WEIGHTS    = [0.40, 0.15, 0.42, 0.03]

# ── Helper: Generate one clean loan record ────────────────────────────────────
def make_loan(loan_id_num):
    grade       = random.choices(GRADES, weights=[20,22,18,15,12,8,5])[0]
    subgrade    = f"{grade}{random.randint(1, 5)}"
    purpose     = random.choice(PURPOSES)
    term        = random.choices(TERMS, weights=[60, 40])[0]
    loan_amount = round(random.randint(10, 400) * 100, 2)

    rate_min, rate_max = {
        "A":(5.0,8.0),"B":(8.5,12.0),"C":(12.5,16.5),
        "D":(17.0,22.0),"E":(22.5,27.0),"F":(27.5,31.5),"G":(32.0,36.0)
    }[grade]
    interest_rate = round(random.uniform(rate_min, rate_max), 2)

    monthly_rate = (interest_rate / 100) / 12
    installment  = round(
        loan_amount * (monthly_rate * (1 + monthly_rate)**term)
        / ((1 + monthly_rate)**term - 1), 2
    )

    issue_date        = fake.date_between(start_date=datetime(2021,1,1),
                                          end_date=datetime(2023,12,31))
    first_payment_date = issue_date + timedelta(days=random.randint(28,35))
    loan_status        = random.choices(LOAN_STATUSES_CLEAN,
                                        weights=STATUS_WEIGHTS)[0]

    if loan_status == "Fully Paid":
        total_payment      = round(installment * term * random.uniform(0.98, 1.02), 2)
        principal_received = loan_amount
        interest_received  = round(total_payment - loan_amount, 2)
        charged_off_amount = 0.0
    elif loan_status == "Charged Off":
        pct_paid           = random.uniform(0.05, 0.60)
        total_payment      = round(loan_amount * pct_paid, 2)
        principal_received = round(total_payment * 0.70, 2)
        interest_received  = round(total_payment * 0.30, 2)
        charged_off_amount = round(loan_amount - principal_received, 2)
    elif loan_status == "Current":
        months_paid        = random.randint(3, term - 1)
        total_payment      = round(installment * months_paid, 2)
        principal_received = round(total_payment * 0.65, 2)
        interest_received  = round(total_payment * 0.35, 2)
        charged_off_amount = 0.0
    elif loan_status in ["Late (31-120 days)", "Default"]:
        months_paid        = random.randint(1, 12)
        total_payment      = round(installment * months_paid, 2)
        principal_received = round(total_payment * 0.70, 2)
        interest_received  = round(total_payment * 0.30, 2)
        charged_off_amount = 0.0
    else:
        total_payment = principal_received = interest_received = charged_off_amount = 0.0

    return {
        "loan_id":             f"LN{loan_id_num:06d}",
        "loan_grade":          grade,
        "loan_subgrade":       subgrade,
        "loan_purpose":        purpose,
        "loan_term_months":    term,
        "loan_amount":         loan_amount,
        "interest_rate":       interest_rate,
        "installment":         installment,
        "issue_date":          issue_date.strftime("%Y-%m-%d"),
        "first_payment_date":  first_payment_date.strftime("%Y-%m-%d"),
        "loan_status":         loan_status,
        "total_payment":       total_payment,
        "principal_received":  principal_received,
        "interest_received":   interest_received,
        "charged_off_amount":  charged_off_amount,
        "credit_score":        random.randint(580, 820),
        "dti":                 round(random.uniform(5.0, 45.0), 2),
        "annual_income":       round(random.randint(250, 2500) * 100, 2),
        "employment_length":   random.choices(
            ["< 1 year","1 year","2 years","3 years","4 years",
             "5 years","6 years","7 years","8 years","9 years","10+ years"],
            weights=[8,8,9,10,9,10,8,8,7,6,17]
        )[0],
        "home_ownership":      random.choices(HOME_OWNERSHIP,
                                              weights=HOME_WEIGHTS)[0],
        "addr_state":          random.choice(VALID_STATES),
    }

# ── Generate clean base records ───────────────────────────────────────────────
print("Generating base loan records...")
loans = [make_loan(i + 1) for i in range(N_CLEAN)]
df    = pd.DataFrame(loans)

# ─────────────────────────────────────────────────────────────────────────────
# INJECT DATA QUALITY ISSUES
# ─────────────────────────────────────────────────────────────────────────────

# [DQ-1] Duplicate rows
dupe_idx = random.sample(range(len(df)), N_DUPES)
df = pd.concat([df, df.iloc[dupe_idx].copy()], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"  [DQ-1] {N_DUPES} duplicate rows injected — total rows: {len(df)}")

# [DQ-2] Null loan amounts
null_amount_idx = random.sample(range(len(df)), 300)
df.loc[null_amount_idx, "loan_amount"] = np.nan
print(f"  [DQ-2] 300 null loan_amount values injected")

# [DQ-3] Negative interest rates
neg_rate_idx = random.sample(range(len(df)), 80)
df.loc[neg_rate_idx, "interest_rate"] = df.loc[neg_rate_idx, "interest_rate"] * -1
print(f"  [DQ-3] 80 negative interest_rate values injected")

# [DQ-4] Invalid credit scores
bad_score_idx = random.sample(range(len(df)), 150)
for i, idx in enumerate(bad_score_idx):
    if i < 75:
        df.loc[idx, "credit_score"] = np.nan
    elif i < 113:
        df.loc[idx, "credit_score"] = random.randint(851, 999)
    else:
        df.loc[idx, "credit_score"] = random.randint(100, 299)
print(f"  [DQ-4] 150 invalid/null credit_score values injected")

# [DQ-5] DTI > 100%
high_dti_idx = random.sample(range(len(df)), 100)
df.loc[high_dti_idx, "dti"] = [round(random.uniform(101, 250), 2) for _ in range(100)]
print(f"  [DQ-5] 100 DTI values > 100% injected")

# [DQ-6] Date sequence errors
date_err_idx = random.sample(range(len(df)), 50)
for idx in date_err_idx:
    issue    = datetime.strptime(df.loc[idx, "issue_date"], "%Y-%m-%d")
    bad_fpd  = issue - timedelta(days=random.randint(1, 10))
    df.loc[idx, "first_payment_date"] = bad_fpd.strftime("%Y-%m-%d")
print(f"  [DQ-6] 50 date sequence errors injected")

# [DQ-7] Mixed casing in loan_status
STATUS_VARIANTS = {
    "Fully Paid":         ["fully paid", "FULLY PAID", "Fully paid"],
    "Charged Off":        ["charged off", "CHARGED OFF", "charged Off"],
    "Current":            ["current", "CURRENT"],
    "Late (31-120 days)": ["late (31-120 days)", "LATE (31-120 DAYS)"],
    "Default":            ["default", "DEFAULT"],
}
mixed_case_idx = random.sample(range(len(df)), 120)
for idx in mixed_case_idx:
    orig    = df.loc[idx, "loan_status"]
    variants = STATUS_VARIANTS.get(orig, [orig.lower()])
    df.loc[idx, "loan_status"] = random.choice(variants)
print(f"  [DQ-7] 120 mixed-case loan_status values injected")

# [DQ-8] Invalid state codes
FAKE_STATES    = ["XX", "ZZ", "QQ", "00", "99", "AB", "BC", "ON"]
bad_state_idx  = random.sample(range(len(df)), 40)
df.loc[bad_state_idx, "addr_state"] = [random.choice(FAKE_STATES) for _ in range(40)]
print(f"  [DQ-8] 40 invalid state codes injected")

# [DQ-9] Null employment_length
null_emp_idx = random.sample(range(len(df)), 200)
df.loc[null_emp_idx, "employment_length"] = np.nan
print(f"  [DQ-9] 200 null employment_length values injected")

# [DQ-10] Grade/subgrade mismatches
mismatch_idx = random.sample(range(len(df)), 60)
for idx in mismatch_idx:
    correct_grade = df.loc[idx, "loan_grade"]
    wrong_grade   = random.choice([g for g in GRADES if g != correct_grade])
    df.loc[idx, "loan_subgrade"] = f"{wrong_grade}{random.randint(1, 5)}"
print(f"  [DQ-10] 60 grade/subgrade mismatches injected")

# ── Export ────────────────────────────────────────────────────────────────────
df.to_csv(f"{output_dir}/loans_raw.csv", index=False)
print(f"\n✅ loans_raw.csv exported — {len(df)} total rows")

In [ ]:
# Cell 5: Generate payment_events_raw.csv
# Monthly payment behavior per loan — used in Q4 (early delinquency signal).
# Generates events for a representative sample of 900 loans.
#
# Intentional data quality issues:
#   [PE-DQ-1] ~30 duplicate payment event records
#   [PE-DQ-2] ~25 payment dates before loan issue date
#   [PE-DQ-3] ~20 zero or negative payment amounts

clean_loan_ids = [f"LN{i+1:06d}" for i in range(N_CLEAN)]
sampled_ids    = random.sample(clean_loan_ids, 900)

payment_events = []
event_id       = 1

for loan_id in sampled_ids:
    loan_row = df[df["loan_id"] == loan_id]
    if loan_row.empty:
        continue

    issue_date_str = loan_row.iloc[0]["issue_date"]
    term           = loan_row.iloc[0]["loan_term_months"]
    installment    = loan_row.iloc[0]["installment"]
    loan_status    = loan_row.iloc[0]["loan_status"]

    try:
        issue_date = datetime.strptime(str(issue_date_str), "%Y-%m-%d")
    except:
        continue

    if loan_status in ["Fully Paid"]:
        n_payments = int(term)
    elif loan_status in ["Charged Off", "Default"]:
        n_payments = random.randint(1, 18)
    elif loan_status == "Current":
        n_payments = random.randint(6, 30)
    else:
        n_payments = random.randint(3, 12)

    for month_num in range(1, n_payments + 1):
        payment_date = issue_date + timedelta(days=30 * month_num)

        if pd.isna(installment) or installment == 0:
            amount_paid = 0.0
        else:
            amount_paid = round(float(installment) * random.uniform(0.95, 1.05), 2)

        if month_num == n_payments and loan_status in ["Charged Off", "Default"]:
            pmt_status = "MISSED"
        elif random.random() < 0.04:
            pmt_status = "LATE"
        else:
            pmt_status = "ON_TIME"

        payment_events.append({
            "event_id":       f"PE{event_id:07d}",
            "loan_id":        loan_id,
            "payment_month":  month_num,
            "payment_date":   payment_date.strftime("%Y-%m-%d"),
            "amount_paid":    amount_paid,
            "payment_status": pmt_status,
        })
        event_id += 1

pe_df = pd.DataFrame(payment_events)

# [PE-DQ-1] Duplicate payment event rows
pe_dupe_idx = random.sample(range(len(pe_df)), 30)
pe_df = pd.concat([pe_df, pe_df.iloc[pe_dupe_idx].copy()], ignore_index=True)
pe_df = pe_df.sample(frac=1, random_state=42).reset_index(drop=True)

# [PE-DQ-2] Payment dates before loan issue date
pe_date_err_idx = random.sample(range(len(pe_df)), 25)
for idx in pe_date_err_idx:
    loan_id  = pe_df.loc[idx, "loan_id"]
    loan_row = df[df["loan_id"] == loan_id]
    if not loan_row.empty:
        issue_date = datetime.strptime(
            str(loan_row.iloc[0]["issue_date"]), "%Y-%m-%d")
        bad_date = issue_date - timedelta(days=random.randint(5, 60))
        pe_df.loc[idx, "payment_date"] = bad_date.strftime("%Y-%m-%d")

# [PE-DQ-3] Zero or negative payment amounts
pe_neg_idx = random.sample(range(len(pe_df)), 20)
for idx in pe_neg_idx:
    pe_df.loc[idx, "amount_paid"] = round(random.uniform(-500, 0), 2)

pe_df.to_csv(f"{output_dir}/payment_events_raw.csv", index=False)
print(f"✅ payment_events_raw.csv exported — {len(pe_df)} total rows")

In [ ]:
# Cell 6: Data generation summary report
# Validate all four output files before downloading.

print("=" * 60)
print("DATA GENERATION SUMMARY — PROJECT 3")
print("=" * 60)

files = {
    "loans_raw.csv":           df,
    "payment_events_raw.csv":  pe_df,
    "loan_grades_lookup.csv":  loan_grades_df,
    "geography_lookup.csv":    geography_df,
}

for fname, frame in files.items():
    print(f"\n📄 {fname}")
    print(f"   Rows: {len(frame):,}  |  Columns: {len(frame.columns)}")
    null_counts = frame.isnull().sum()
    nulls = null_counts[null_counts > 0]
    if not nulls.empty:
        for col, cnt in nulls.items():
            pct = round(cnt / len(frame) * 100, 1)
            print(f"   ⚠  {col}: {cnt} nulls ({pct}%)")

print("\n" + "=" * 60)
print("KNOWN DATA QUALITY ISSUES — loans_raw.csv")
print("=" * 60)
issues = [
    ("[DQ-1]",  "100 duplicate loan records (2.7%)"),
    ("[DQ-2]",  "~300 null loan amounts (8.1%)"),
    ("[DQ-3]",  "~80 negative interest rates (2.2%)"),
    ("[DQ-4]",  "~150 invalid/null credit scores (4.1%)"),
    ("[DQ-5]",  "~100 DTI values over 100% (2.7%)"),
    ("[DQ-6]",  "~50 date sequence errors (1.4%)"),
    ("[DQ-7]",  "~120 mixed-case loan_status values (3.2%)"),
    ("[DQ-8]",  "~40 invalid state codes (1.1%)"),
    ("[DQ-9]",  "~200 null employment_length values (5.4%)"),
    ("[DQ-10]", "~60 grade/subgrade mismatches (1.6%)"),
]
for code, desc in issues:
    print(f"  {code}  {desc}")

print(f"\n✅ All files saved to: {output_dir}")